# API Exploration: Open Food Facts, Open Beauty Facts, Open Product Facts #

## 1. Imports and Settings ##

In [1]:
import requests
import pandas as pd
import numpy as np
import time
from collections import Counter

In [2]:
HEADERS = {
    "User-Agent": "WBS-ESG-Analyzer/1.0 (student project)",
    "Accept": "application/json",
    "Accept-Language": "en-US,en;q=0.9",
}

In [3]:
SOURCES = {
    "food": "https://world.openfoodfacts.org",
    "beauty": "https://world.openbeautyfacts.org",
    "products": "https://world.openproductsfacts.org"
}

## 2. Define Useful Fields ##

In [4]:
USEFUL_FIELDS = [
    "code",
    "product_name",
    "generic_name",
    "brands",
    "categories",
    "categories_tags",
    "labels",
    "labels_tags",
    "ingredients_text",
    "ingredients_tags",
    "packaging",
    "packaging_tags",
    "packagings",
    "countries",
    "countries_tags",
    "stores",
    "origins",
    "origins_tags",
    "manufacturing_places",
    "ecoscore_grade",
    "ecoscore_score",
    "image_url",
    "last_modified_t",
    "created_t"
]

FIELDS_PARAM = ",".join(USEFUL_FIELDS)

## 3. Germany Filtering Setup ##

In [5]:
GERMANY_COUNTRY_TAG = "en:germany"

In [6]:
def normalize_tags(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str) and value.strip():
        return [tag.strip() for tag in value.split(",")]
    return []

In [7]:
def is_german_product(product):
    country_tags = normalize_tags(product.get("countries_tags"))
    return GERMANY_COUNTRY_TAG in country_tags

## 4. API Helper Functions ##

In [8]:
def request_json(url, params=None, retries=5, timeout=60):
    for attempt in range(1, retries + 1):
        response = requests.get(url, params=params, headers=HEADERS, timeout=timeout)

        if response.status_code == 200:
            return response.json()

        if response.status_code in (429, 503) and attempt < retries:
            wait_seconds = 3 * attempt
            print(f"Temporary status {response.status_code}; retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)
            continue

        print("Failed URL:", response.url)
        response.raise_for_status()

Open Food Facts occasionally returned temporary 503 responses during search. Retry logic was necessary and successful.

In [9]:
# Search Products

def search_products(source_name, query, page_size=100, page=1):
    base_url = SOURCES[source_name]
    url = f"{base_url}/cgi/search.pl"

    params = {
        "search_terms": query,
        "search_simple": 1,
        "action": "process",
        "json": 1,
        "page_size": page_size,
        "page": page,
        "fields": FIELDS_PARAM,
    }

    data = request_json(url, params=params)
    return data.get("products", [])

In [10]:
# Retrieve by Barcode

def get_product_by_barcode(source_name, barcode):
    base_url = SOURCES[source_name]
    url = f"{base_url}/api/v3/product/{barcode}"

    params = {
        "fields": FIELDS_PARAM
    }

    return request_json(url, params=params)

## 5. Retrieve Product Samples ##

In [11]:
queries = {
    "food": [
        "schokolade",
        "muesli",
        "milch",
        "kaffee"
    ],
    "beauty": [
        "shampoo",
        "seife",
        "cream",
        "deo"
    ],
    "products": [
        "handy",
        "batterie",
        "waschmittel",
        "toothbrush"
    ]
}

Because the dataset is filtered for products available in Germany, German search terms generally return more relevant results. However, Open*Facts product names and categories are multilingual, so some English terms perform better. We therefore use German search terms where they produce strong results and English fallback terms where German terms return few or no products.

In [12]:
# Fetch and filter

all_products = []

for source, query_list in queries.items():
    for query in query_list:
        products = search_products(source, query, page_size=100)
        products = [p for p in products if is_german_product(p)]

        for product in products:
            product = product.copy()
            product["source"] = source
            product["search_query"] = query
            all_products.append(product)

        print(source, query, len(products))
        time.sleep(1)

Temporary status 503; retrying in 3s...
food schokolade 93
food muesli 10
food milch 93
food kaffee 96
beauty shampoo 12
beauty seife 93
beauty cream 6
beauty deo 38
products handy 7
products batterie 27
products waschmittel 49
products toothbrush 15


In [13]:
barcode_test = get_product_by_barcode("food", "3017624010701")
barcode_test["product"]

{'brands': 'Ferrero',
 'categories': 'Cocoa and hazelnuts spreads, de:Other',
 'categories_tags': ['en:breakfasts',
  'en:spreads',
  'en:sweet-spreads',
  'fr:pates-a-tartiner',
  'en:hazelnut-spreads',
  'en:chocolate-spreads',
  'en:cocoa-and-hazelnuts-spreads',
  'de:Other'],
 'code': '3017624010701',
 'countries': 'France, United Kingdom',
 'countries_tags': ['en:france', 'en:united-kingdom'],
 'created_t': 1602617601,
 'ecoscore_grade': 'd',
 'ecoscore_score': 35,
 'generic_name': '',
 'image_url': 'https://images.openfoodfacts.org/images/products/301/762/401/0701/front_en.100.400.jpg',
 'ingredients_tags': ['de:sugar',
  'de:palm-oil',
  'de:hazelnuts',
  'de:skimmed-milk-powder',
  'de:fat-reduced-cocoa',
  'de:emulsifier',
  'en:vanillin'],
 'ingredients_text': 'sugar, palm oil, hazelnuts, skimmed milk powder, fat reduced cocoa, emulsifier, vanillin.',
 'labels': 'No gluten',
 'labels_tags': ['en:no-gluten'],
 'last_modified_t': 1778846121,
 'manufacturing_places': '',
 'origi

### Create raw dataframe ###

In [14]:
raw_sample_df = pd.DataFrame(all_products)

print(raw_sample_df.shape)
display(raw_sample_df["source"].value_counts())

# Save raw data

raw_sample_df.to_csv("raw_openfacts_germany_sample.csv", index=False)

(539, 27)


source
food        292
beauty      149
products     98
Name: count, dtype: int64

## 6. Remove Duplicates ##

In [15]:
df = raw_sample_df.drop_duplicates(subset=["source", "code"]).copy()

print("Before deduplication:", raw_sample_df.shape)
print("After deduplication:", df.shape)

df["source"].value_counts()

Before deduplication: (539, 27)
After deduplication: (539, 27)


source
food        292
beauty      149
products     98
Name: count, dtype: int64

No Duplicates found.

## 7. Clean Text Fields ##

In [16]:
text_columns = [
    "product_name",
    "generic_name",
    "brands",
    "categories",
    "labels",
    "ingredients_text",
    "packaging",
    "countries",
    "stores",
    "origins",
    "manufacturing_places"
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

## 8. Normalize Tag/List Fields ##

In [17]:
tag_columns = [
    "categories_tags",
    "labels_tags",
    "ingredients_tags",
    "packaging_tags",
    "countries_tags",
    "origins_tags"
]

for col in tag_columns:
    if col in df.columns:
        df[col] = df[col].apply(normalize_tags)

## 9. Create Data Completeness Flags ##

In [18]:
def has_list_data(value):
    return isinstance(value, list) and len(value) > 0

In [19]:
df["has_product_name"] = df["product_name"].str.len() > 0
df["has_brand"] = df["brands"].str.len() > 0
df["has_categories"] = df["categories_tags"].apply(has_list_data)
df["has_labels"] = df["labels_tags"].apply(has_list_data)
df["has_ingredients"] = (
    df["ingredients_text"].str.len() > 0
) | (
    df["ingredients_tags"].apply(has_list_data)
)
df["has_packaging"] = (
    df["packaging"].str.len() > 0
) | (
    df["packaging_tags"].apply(has_list_data)
)
df["has_country"] = df["countries_tags"].apply(has_list_data)
df["has_image"] = df["image_url"].fillna("").astype(str).str.len() > 0
ecoscore_grade_clean = (
    df["ecoscore_grade"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

df["has_ecoscore"] = ~ecoscore_grade_clean.isin(["", "unknown", "not-applicable"])

## 10. Create a Clean Dataset ##

In [20]:
clean_columns = [
    "source",
    "search_query",
    "code",
    "product_name",
    "generic_name",
    "brands",
    "categories",
    "categories_tags",
    "labels",
    "labels_tags",
    "ingredients_text",
    "ingredients_tags",
    "packaging",
    "packaging_tags",
    "countries",
    "countries_tags",
    "stores",
    "origins",
    "origins_tags",
    "manufacturing_places",
    "ecoscore_grade",
    "ecoscore_score",
    "image_url",
    "last_modified_t",
    "created_t",
    "has_product_name",
    "has_brand",
    "has_categories",
    "has_labels",
    "has_ingredients",
    "has_packaging",
    "has_country",
    "has_image",
    "has_ecoscore"
]

clean_columns = [col for col in clean_columns if col in df.columns]

clean_sample_df = df[clean_columns].copy()

In [21]:
clean_sample_df.sample(15)

,source,search_query,code,product_name,generic_name,brands,categories,categories_tags,labels,labels_tags,...,created_t,has_product_name,has_brand,has_categories,has_labels,has_ingredients,has_packaging,has_country,has_image,has_ecoscore
322,beauty,seife,4056489762027,Seife Soap Province Honey,,Cien,"Incorrect product type, non-food-products, ope...","[en:incorrect-product-type, en:non-food-produc...",,[],...,1732032712,True,True,True,False,False,False,True,True,False
334,beauty,seife,4019886190060,Seife,,sodasan,"Incorrect product type, non-food-products, ope...","[en:incorrect-product-type, en:non-food-produc...",,[],...,1696346085,True,True,True,False,False,False,True,False,False
446,products,handy,0888462062367,"iPhone 6, space gray, 16GB",,Apple,"Mobile Phones, iPhone smartphones, Communicati...","[en:electronics, en:communications, en:telepho...",,[],...,1611580797,True,True,True,False,False,True,True,True,False
220,food,kaffee,4056489010241,Kaffee Bio gold Kaffee,,Bellarom,Arabica coffees,"[en:plant-based-foods-and-beverages, en:plant-...","non-EU Agriculture, EG-Öko-Verordnung, DE-ÖKO-003","[en:organic, en:eu-organic, en:non-eu-agricult...",...,1590228679,True,True,True,True,True,False,True,True,False
77,food,schokolade,4061458022002,Nussknacker - Zartbitterschokolade,,"Aldi, Choceur, WIHA","Nuts and their products, Milk chocolates, Dark...","[en:plant-based-foods-and-beverages, en:plant-...","Sustainable farming, Made in Germany, Nutrisco...","[en:sustainable-farming, en:made-in-germany, e...",...,1564077114,True,True,True,True,True,True,True,True,True
240,food,kaffee,8711000506448,Kaffeesticks Espresso,,Jacobs,"Beverages, Coffees, de:Instant, espresso, pfla...","[en:beverages-and-beverages-preparations, en:p...",,[],...,1569770006,True,True,True,False,True,True,True,True,False
273,food,kaffee,4008167154679,Espresso d'oro,,Dallmayr,"Beverages, Roasted coffee beans","[en:beverages-and-beverages-preparations, en:p...",,[],...,1562685769,True,True,True,False,False,True,True,True,True
171,food,milch,4335619096448,Soja Vanille Geschmack,,Vemondo,Soy milk yogurts,"[en:plant-based-foods-and-beverages, en:fermen...","Vegan, FSC Mix, Nutriscore Grade B","[en:vegetarian, en:vegan, en:fsc, en:fsc-mix, ...",...,1728506816,True,True,True,True,True,False,True,True,True
358,beauty,seife,7630028629377,Seife,,,"Incorrect product type, non-food-products, ope...","[en:incorrect-product-type, en:non-food-produc...",,[],...,1729079572,True,False,True,False,False,False,True,False,False
81,food,schokolade,5000159471510,M&M's Erdnuss,Schokolinsen mit Erdnüssen,M&M'S,Chocolate covered peanuts,"[en:plant-based-foods-and-beverages, en:plant-...",Green Dot,[en:green-dot],...,1491852170,True,True,True,True,True,True,True,True,True


Some text fields contain encoding artifacts or mixed-language values. This should be handled in later preprocessing if text quality becomes important.

In [22]:
# Save Clean DataFrame

clean_sample_df.to_csv("clean_openfacts_germany_sample.csv", index=False)

## 11. Explore Dataset Size ##

In [23]:
clean_sample_df.shape

(539, 34)

In [24]:
clean_sample_df["source"].value_counts()

source
food        292
beauty      149
products     98
Name: count, dtype: int64

In [25]:
clean_sample_df.groupby("source")["code"].nunique()

source
beauty      149
food        292
products     98
Name: code, dtype: int64

## 12. Explore Data Completeness ##

In [26]:
completeness_columns = [
    "has_product_name",
    "has_brand",
    "has_categories",
    "has_labels",
    "has_ingredients",
    "has_packaging",
    "has_country",
    "has_image",
    "has_ecoscore"
]

completeness = (
    clean_sample_df
    .groupby("source")[completeness_columns]
    .mean()
    .round(3)
)

completeness

,has_product_name,has_brand,has_categories,has_labels,has_ingredients,has_packaging,has_country,has_image,has_ecoscore
source,,,,,,,,,
beauty,0.987,0.785,0.852,0.215,0.289,0.168,1.0,0.664,0.013
food,0.986,0.990,1.000,0.863,0.952,0.675,1.0,0.997,0.716
products,1.000,0.929,0.612,0.082,0.051,0.214,1.0,0.602,0.000


Food:     strong categories, ingredients, labels, Eco-Score

Beauty:   decent categories, weaker ingredients/labels, almost no usable Eco-Score

Products: sparse ingredients/labels, no usable Eco-Score

## 13. Explore Top Categories, Labels, Packaging ##

In [27]:
def top_tags(df, source, column, n=20):
    values = []

    subset = df[df["source"] == source]

    for tags in subset[column]:
        values.extend(normalize_tags(tags))

    return pd.DataFrame(
        Counter(values).most_common(n),
        columns=["tag", "count"]
    )

In [28]:
top_tags(clean_sample_df, "food", "categories_tags", 20)

,tag,count
0,en:plant-based-foods-and-beverages,146
1,en:plant-based-foods,127
2,en:beverages-and-beverages-preparations,103
3,en:beverages,98
4,en:snacks,79
5,en:sweet-snacks,79
6,en:coffees,75
7,en:fermented-foods,62
8,en:cocoa-and-its-products,60
9,en:dairies,60


In [29]:
top_tags(clean_sample_df, "beauty", "labels_tags", 20)

,tag,count
0,en:vegetarian,21
1,en:vegan,21
2,en:the-vegan-society,7
3,en:made-in-germany,5
4,en:without-microplastics,4
5,en:without-silicon,2
6,en:green-point,2
7,de:Recyceltes Plastik,1
8,en:no-colorings,1
9,en:pefc,1


In [30]:
top_tags(clean_sample_df, "products", "packaging_tags", 20)

,tag,count
0,pappe,8
1,kunststoff,5
2,kunststoffbeutel,2
3,papier,1
4,de:produkt,1
5,fr:produkt,1
6,blister,1
7,sichtverpackung,1
8,pet,1
9,pap,1


## 14. Explore Eco-Score for German Food Products ##

In [31]:
food_df = clean_sample_df[clean_sample_df["source"] == "food"]

food_df["ecoscore_grade"].value_counts(dropna=False)

ecoscore_grade
unknown    83
d          45
a          36
c          32
b          30
f          30
e          25
a-plus     11
Name: count, dtype: int64

In [32]:
food_df[[
    "product_name",
    "brands",
    "countries",
    "ecoscore_grade",
    "ecoscore_score"
]].dropna().head(10)

,product_name,brands,countries,ecoscore_grade,ecoscore_score
1,Dunkle Schokolade mit ganzen Haselnüssen,fin CARRE,"Belgium, France, Germany, Netherlands, Poland,...",a,75.0
2,Hazelnut Milk Chocolate,fin CARRE,"Bulgaria, France, Germany",c,58.0
4,Dunkle Ganze Mandel,"Fin carré, Lidl","France, Germany",c,58.0
5,"Eat Natural Vegan - Erdnüsse, Kokos & dunkle S...","Eat Natural, Ferrero","France, Germany",c,48.0
6,Chocolate Fudge Brownie,Ben & Jerry's,"Austria, France, Germany, Netherlands, Spain, ...",b,68.0
7,Milka Choc & Choc,Milka,"Belgium, France, Germany, Spain",d,32.0
8,Magnum Mini Almond,MAGNUM,"France, Germany",a-plus,91.0
9,"Lebkuchen Hearts, Pretzels & Stars Zartbitter",Favorina,"France, Germany",b,68.0
10,Schoko Crunchy,dmBio,"France, Germany, Romania",b,74.0
11,Eis Mandel,"Bon Gelati, Lidl, Lidl Bon Gelati","Bulgaria, France, Germany, United Kingdom",a,83.0


In [33]:
# Check Eco-Score availability by source

clean_sample_df.groupby("source")["has_ecoscore"].mean()

source
beauty      0.013423
food        0.715753
products    0.000000
Name: has_ecoscore, dtype: float64

Eco-Score is useful mainly for Open Food Facts. It is not reliable as a universal score across all sources.

## 15. Inspect Real Rows ##

In [34]:
display_columns = [
    "source",
    "product_name",
    "brands",
    "categories",
    "labels",
    "ingredients_text",
    "packaging",
    "countries",
    "ecoscore_grade"
]

clean_sample_df[display_columns].sample(10)

,source,product_name,brands,categories,labels,ingredients_text,packaging,countries,ecoscore_grade
189,food,Crefee mit feinen Kräutern,Milbona,"Cream cheeses, de:Frischkäsezubereitung",Made in Germany,"Frischkäse, Speisesalz, Kräuter, Gewürze, Knob...","Metall,Kunststoff,PET - Polyethylenterephtalat...","Germany, Hungary, Lithuania, Romania, Serbia, ...",d
270,food,Nescafé Classic Mild Instantkaffee,"Nescafe, Nescafé","Beverages, Coffees, Instant coffees",,nescafe,Glass,"France, Germany",unknown
169,food,Hafermilch Voll,Oatly,Oat-based drinks,"No milk, Vegan, FSC, Green Dot, No soy","Wasser, HAFER 10%, Rapsöl, Säureregulator (Dik...",,"Austria, Germany, Switzerland",c
56,food,Schokolade Dunkle Voll-Nuss,Ritter Sport,"Dark chocolate bar with less than 70% cocoa, D...","Sustainable farming, Green Dot, Rainforest All...","Zucker, Kakaomasse, _Haselnüsse_ 25%, Kakaobut...","Kunststoff,Folie",Germany,d
495,products,Frosch Aloe Vera Waschmittel,Werner u. Mertz,,,,,Germany,NaN
300,beauty,Shampoo vegan Aloe Vera,Cien,"Incorrect product type, non-food-products, ope...",Vegan,,,Germany,unknown
47,food,Edelbitter- Schokolade,"Aldi, Gut Bio, Halba",Dark chocolates,"Fairtrade International, EG-Öko-Verordnung, CH...","Kakaomasse¹, Rohrzucker¹, Kakaobutter¹. ¹aus k...",,Germany,d
331,beauty,Haar Seife Speick,Speick,"Incorrect product type, non-food-products, ope...",,"Ingredients: Sodium Palmate, Sodium Cocoate, A...",,Germany,unknown
304,beauty,Balea feste Seife,Balea,"Incorrect product type, non-food-products, ope...",,,,Germany,unknown
287,food,Cappuccino,Jacobs,"Beverages, Coffees, Powdered cappucino",,"Zucker, Glukosesirup, Kokosfett (ganz gehärtet...",,"France, Germany",unknown


### Best rows for later scoring ###

In [35]:
clean_sample_df[
    clean_sample_df["has_product_name"]
    & clean_sample_df["has_brand"]
    & clean_sample_df["has_categories"]
    & clean_sample_df["has_labels"]
][display_columns].head(10)

,source,product_name,brands,categories,labels,ingredients_text,packaging,countries,ecoscore_grade
1,food,Dunkle Schokolade mit ganzen Haselnüssen,fin CARRE,fr:Barres chocolatées à la noix de coco,"Sustainable farming, Fairtrade cocoa, Made in ...","Kakaomasse, Zucker, Haselnusskerne, Kakaobutte...","Plastique,PP 5 - Polypropylène","Belgium, France, Germany, Netherlands, Poland,...",a
2,food,Hazelnut Milk Chocolate,fin CARRE,Milk chocolates with hazelnuts,"Fairtrade cocoa, Green Dot, Max Havelaar, UTZ ...","sugar, cocoa butter, 12% hazelnuts, skimmed mi...","Film,Plastique","Bulgaria, France, Germany",c
3,food,Toffifee weiße Schokolade,"Storck, Storck KG",Bonbons,"Green Dot, Limited edition","Zucker, pflanzliche Fette (Palm, Shea), Haseln...","Plastic,Cardboard,Box,Film,Paperboard,Tray","Austria, Bulgaria, Czech Republic, France, Ger...",unknown
4,food,Dunkle Ganze Mandel,"Fin carré, Lidl",Dark chocolates with almonds,"Fairtrade cocoa, FSC Mix, Green Dot, Max Havel...","Zucker, Kakaomasse, 25% Mandeln, Kakaobutter, ...","Carton, Boite carton de 11g, FSC C021442, FSC ...","France, Germany",c
5,food,"Eat Natural Vegan - Erdnüsse, Kokos & dunkle S...","Eat Natural, Ferrero","Chocolate cereal bars, Fruits cereal bars","No gluten, Vegan, FSC Mix","Bitterschokolade 20 % (Kakaomasse, Rohrzucker,...",,"France, Germany",c
6,food,Chocolate Fudge Brownie,Ben & Jerry's,Chocolate ice cream tubs,"Vegetarian, Halal, Triman","_RAHM_ 25%, Trinkwasser, Zucker, konzentrierte...","Card-lid,Card-tub,Green Dot,Couvercle carton,F...","Austria, France, Germany, Netherlands, Spain, ...",b
7,food,Milka Choc & Choc,Milka,"Chocolate candies, Chocolate cakes, de:Kuchen ...",Green Dot,"Zucker, _Weizenmehl_, Glukose-Fruktosesirup, _...","Plastic,PP 5 - Polypropylene,Bag,Tray","Belgium, France, Germany, Spain",d
8,food,Magnum Mini Almond,MAGNUM,Vanilla ice cream bars coated with milk chocol...,"No gluten, Rainforest Alliance Cocoa, Triman","Entrahmte MILCH, Zucker, Kakaobutter', Trinkwa...",,"France, Germany",a-plus
9,food,"Lebkuchen Hearts, Pretzels & Stars Zartbitter",Favorina,"Biscuits, Plain gingerbreads covered with choc...","Vegan, Fairtrade cocoa, Green Dot, Made in Ger...","Weizenmehl, Glukose-Fruktose-Sirup, 25% Zartbi...","Plastic, Cardboard, Pp-tub, fr:Point vert, nl:...","France, Germany",b
10,food,Schoko Crunchy,dmBio,Mueslis with chocolate,Fairtrade International,"48% HAFERFLOCKEN, Reissirup, Rapsöl, WEIZENFLO...",,"France, Germany, Romania",b


### Weak rows ###

In [36]:
clean_sample_df[
    ~clean_sample_df["has_product_name"]
    | ~clean_sample_df["has_categories"]
][display_columns].head(10)

,source,product_name,brands,categories,labels,ingredients_text,packaging,countries,ecoscore_grade
21,food,,Schogetten,Milk chocolates with hazelnuts,"Green Dot, New recipe","Zucker, Kakaobutter, Sahnepulver, Haselnüsse (...","Pappe, en:Container, 21 PAP, 21","Bulgaria, Denmark, France, Germany, Greece, Is...",d
102,food,,Crownfield,Mueslis with fruits,"High fibres, Nutriscore","45% mistura de frutos [14,5% sultanas (sultana...",en:07 inne,"Belgium, France, Germany, Netherlands, Poland,...",b
165,food,,MILSLANI,Skyrs,"Ohne Gentechnik, Made in Germany, Nutriscore G...","Magermilch, Milchsäurekulturen (enthalten Milc...",,Germany,b
227,food,,Barissimo,"Beverages, Instant coffees",,,,"France, Germany",unknown
295,beauty,Anti Schuppen Shampoo,head & shoulders,,,,,Germany,unknown
297,beauty,,Garnier,"Shampoos, Incorrect product type, non-food-pro...",,,,"Germany, Iraq",unknown
299,beauty,shampoo,,,,,,Germany,unknown
302,beauty,Anti schuppen Shampoo,Cien,,,,,Germany,unknown
310,beauty,Pfirsichblüte Sensitiv Seife,Frosch,,,"INGREDIENTS * Aqua, Sodium Laureth Sulfate, So...",,Germany,unknown
329,beauty,Frosch Aloe Vera Creme Seife Refill,,,,,,"Germany, Ireland",unknown


## 16. Final Markdown Conclusion ##

Day 1 Findings:

We successfully retrieved Germany-specific product data from Open Food Facts, Open Beauty Facts, and Open Products Facts.

The Germany filter is based on `countries_tags`, meaning the products are listed as available in Germany. This does not necessarily mean they are manufactured in Germany.

After filtering and duplicate removal, we created a clean dataset containing product names, brands, categories, labels, ingredients, packaging, countries, and available Eco-Score data.

Open Food Facts appears to provide the richest structured data for German products, especially for food categories, ingredients, labels, and Eco-Score. Open Beauty Facts provides useful brand, ingredient, and label data, but no consistent Eco-Score. Open Products Facts is more sparse and should be treated carefully.

The dataset used on Day 1 is a Germany-filtered sample, not a full extract of the Open*Facts databases. The goal was to test API accessibility, field availability, data cleaning, and source reliability. A larger paginated extraction can be implemented later if needed.